## DATA PIPELINE


In [1]:
import time
from pathlib import Path
import requests
import re
import csv
from collections import Counter
from __future__ import annotations
import pandas as pd
import numpy as np


#helper modules
from Python_code.modules import json_parser_theo as jp, classify_chats as cc, paser_chatti_html as hp, mail_parser as mp

#pfade anlegen
repo = Path(".").resolve().parent
path_raw     = repo / "data/raw/data_sosci"
path_uploads = path_raw / "uploads"
csv_path     = path_raw / "daten.csv"
mapping_path = path_raw / "chatlog_mapping.csv"
uploads_root = path_raw / "uploads"
path_proc    = repo / "data/processed"
path_mail    = path_raw / "Mail"
path_other   = path_raw / "other"
####################################
###Verwendung der Sosci APIs########
###################################

""" Um die daten abzurufen wird eine simple API verwednet, die nur für das data retrival einens Project funktioniert. Diese braucht für die Authentifizierung nur eine url (hier mit path_sosci_key gespeichert.
Um die hochgeladenen Datein herunter zu laden muss man die REST-API verwenden diese ist Personenspezifisch und braucht einen header (mit key) und url. Mt dieser API können datein einzeld herunter geladen werden und automatische in eine Ordnerstruktur eingebunden werden, die über ein Mapping festgehalten wird, für spätere verarbeitungensschritte"""

#pfade für keys
path_sosci_key        = repo / "API_KEYS/sosci_data_key.txt"
path_sosci_per_header = repo / "API_KEYS/sosci_per_header.txt"

#laden der url inclusive key
sosci_data_url = path_sosci_key.read_text().strip()

# Header-Wert robust einlesen
raw_header = path_sosci_per_header.read_text().strip()
if raw_header.lower().startswith("authorization:"):
    raw_header = raw_header.split(":", 1)[1].strip()

# Header definieren und urls
headers = {"Authorization": raw_header}
BASE = "https://survey.ifkw.lmu.de/admin/api.php?v1"
uploads_list_url = f"{BASE}/projects/5393/uploads" #wenn man ein / am Ende einfügt erzeugt das ein fehler

#CSV API-Abfrage
response = requests.get(sosci_data_url)
response.raise_for_status()    #Extrahiert den Chat-Verlauf und gibt ihn als csv zurück.
csv_path.write_bytes(response.content)
print(f"CSV gespeichert unter: {csv_path}")


#uploads-api-abfrage
r = requests.get(uploads_list_url, headers=headers)
r.raise_for_status()
filenames = r.json()["files"]
print(f"{len(filenames)} hochgeladene Dateien gefunden.")

#pattern um alle wichtigen infos aus html name zu ziehen
pattern = re.compile(r"^([A-Za-z]+\d+)\.(\d+)\.(\w+)$")
mapping_rows = []

for fname in filenames:
    match = pattern.match(fname)
    if not match:
        print(f"Dateiname passt nicht zum erwarteten Muster: {fname}")
        continue

    frage_code, teilnehmer_id, ext = match.groups()
    teilnehmer_id = str(int(teilnehmer_id))
    # Unterordner pro Person anlegen: uploads/<teilnehmer_id>/<fname>
    # wird später für den html_paser verwendent
    person_dir = path_uploads  / teilnehmer_id
    person_dir.mkdir(parents=True, exist_ok=True)
    out_path = person_dir / fname

    try:
        file_response = requests.get(f"{uploads_list_url}{"/"}{fname}", headers=headers)
        file_response.raise_for_status()
        out_path.write_bytes(file_response.content)
        print(f"Gespeichert: {out_path}")
        #mapping für weiterverarbeitung erweitern
        mapping_rows.append({"teilnehmer_id": teilnehmer_id, "frage_code": frage_code,
                              "dateiname": fname, "pfad": str(out_path)})
     #Fehler ausgeben statt abrechen, da wenn viel daten, einfach weitergearbeite wird
    except requests.exceptions.RequestException as e:
        print(f"Fehler bei {fname}: {e}")

    time.sleep(0.2) #sleep timer für den armen server

#mapping speicher, für weiterverarbeitung

with open(mapping_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["teilnehmer_id", "frage_code", "dateiname", "pfad"])
    writer.writeheader()
    writer.writerows(mapping_rows)

print(f"\nZuordnungstabelle gespeichert unter: {mapping_path}")

#Kontrolle wievele vhats per person
counts = Counter(row["teilnehmer_id"] for row in mapping_rows)
print("\nChats pro Teilnehmer-ID:")
for tid, n in sorted(counts.items()):
    print(f"  {tid}: {n} Datei(en)")

CSV gespeichert unter: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/daten.csv
23 hochgeladene Dateien gefunden.
Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/255/CH03.000255.html
Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/263/CH03.000263.html
Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/264/CH03.000264.html
Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/296/CH03.000296.html
Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/368/CH03.000368.html
Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/255/CH10.000255.html
Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/263/CH10.000263.html
Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_so

In [2]:
# Vorschau der Rohdaten
df_raw = pd.read_csv(csv_path, sep="\t")

In [3]:

################
# Survey-CSV bereinigen (Zeit-/Metavariablen entfernen)
###############
df_raw = pd.read_csv(csv_path, sep="\t")

drop_exact = ["MAILSENT", "LASTDATA", "Q_VIEWER", "LASTPAGE", "MAXPAGE",
              "MODE", "STARTED", "REF", "QUESTNNR"]
time_cols  = [c for c in df_raw.columns if c.startswith("TIME")]   # TIME001..TIME_SUM
drop_cols  = [c for c in (drop_exact + time_cols) if c in df_raw.columns]

df_survey_clean = df_raw.drop(columns=drop_cols)
df_survey_clean.to_csv(path_proc / "survey_clean.csv", index=False)
print(f"Survey bereinigt: {df_raw.shape[1]} -> {df_survey_clean.shape[1]} Spalten "
      f"({len(drop_cols)} entfernt)")


#######################################################
# Missing which occured because of too big data junks #
# and were sent to us per mail                        #
######################################################

# Von SoSci gelöschte Angaben (Interview > 64 KB) aus den .eml-Mails
# in df_raw zurückschreiben, bevor die JSON-Spalten transformiert werden

#händische ergänzugen die ohne caseid geschickt wurden
#aber mit uhrzeit und fragen missing rekonstruiert werden konnten
with open(path_other / "ergänzung.txt", "r", encoding="utf-8") as f:
    df_raw.loc[df_raw["CASE"].astype(int) == 339, "CH08s"] = f.read()

with open(path_other / "ergänzung2.txt", "r", encoding="utf-8") as f:
    df_raw.loc[df_raw["CASE"].astype(int) == 367, "CH09s"] = f.read()
df_raw = mp.apply_mail_values(df_raw, path_mail)


######################################################
# Alle Chats parsen (JSON aus CSV + HTML-Uploads)    #
# Gültigkeit basiert auf tatsächlich extrahierten    #
# Chats, nicht nur auf ausgefüllten Zellen           #
######################################################

JSON_INPUT_COLS = ["CH02s", "CH06s", "CH07s", "CH08s", "CH09s"]  # Chat als JSON-Direkteingabe
UPLOAD_COLS     = ["CH03", "CH10", "CH11", "CH12", "CH13"]       # Chat als HTML-Upload (2 = hochgeladen)

# JSONs aus der CSV (Export-Prompt wird im Parser gefiltert)
rows = []
for _, r in df_raw.iterrows():
    tid = str(int(r["CASE"]))
    for col in JSON_INPUT_COLS:
        if col not in df_raw.columns or pd.isna(r[col]) or not str(r[col]).strip():
            continue
        messages = jp.extract_messages(str(r[col]))
        if not messages:
            print(f"WARNUNG: CASE {tid} {col}: JSON vorhanden, aber keine Nachrichten extrahierbar")
            continue
        for turn, m in enumerate(messages, start=1):
            rows.append({
                "teilnehmer_id": tid,
                "frage_code":    col[:-1],          # CH02s -> CH02
                "quelle":        "json_csv",        # zum Überprüfen: html oder json
                "turn":          turn,
                "role":          m["role"],
                "content":       m["content"],
            })

# HTMLs aus den Uploads (via Mapping) — nur Dateien, deren Upload-Indikator
# in der Survey-CSV auch gesetzt ist (== 2)
mapping = pd.read_csv(mapping_path, dtype={"teilnehmer_id": str})
survey_cases = set(df_raw["CASE"].astype(int))
for _, r in mapping.iterrows():
    tid  = str(int(r["teilnehmer_id"]))
    code = r["frage_code"]
    if int(tid) not in survey_cases:
        print(f"WARNUNG: Upload {r['dateiname']} ohne zugehörigen Survey-Fall — übersprungen")
        continue
    flag = df_raw.loc[df_raw["CASE"].astype(int) == int(tid), code]
    if not flag.eq(2).any():
        print(f"WARNUNG: CASE {tid} {code}: Datei hochgeladen, aber Indikator != 2 — übersprungen")
        continue
    html_path = uploads_root / tid / r["dateiname"]
    if not html_path.exists():
        print(f"HTML fehlt: {html_path}")
        continue
    messages = hp.extract(html_path)
    if not messages:
        print(f"WARNUNG: CASE {tid} {code}: HTML enthält keine Chat-Nachrichten "
              f"(vermutlich leere ChatGPT-Seite gespeichert) — {r['dateiname']}")
        continue
    for turn, m in enumerate(messages, start=1):
        rows.append({
            "teilnehmer_id": tid,
            "frage_code":    code,
            "quelle":        "html_upload",
            "turn":          turn,
            "role":          m["speaker"],          # HTML-Parser: "speaker"/"text"
            "content":       m["text"],
        })

df_all_chats = pd.DataFrame(rows)

######################################################
# Ausschluss: nur Fälle mit >= 5 tatsächlich         #
# extrahierten Chats (JSON und HTML zählen zusammen) #
######################################################

chats_per_person = (df_all_chats.groupby("teilnehmer_id")["frage_code"]
                    .nunique().sort_index())
valid_cases = set(chats_per_person[chats_per_person >= 5].index.astype(int))
excluded    = sorted(set(df_raw["CASE"].astype(int)) - valid_cases)

print(f"\nChats pro Person (tatsächlich extrahiert):")
for tid, n in chats_per_person.items():
    marker = "" if int(tid) in valid_cases else "   <-- ausgeschlossen (< 5)"
    print(f"  {tid}: {n} Chat(s){marker}")
print(f"\nWeiterverarbeitung: {len(valid_cases)} von {len(df_raw)} Fällen mit >= 5 gültigen Chats")
print("Ausgeschlossen:", excluded)

df_raw = df_raw[df_raw["CASE"].astype(int).isin(valid_cases)].copy()

###############################################################
# DATENSATZ 2: Chat-Nachrichten der gültigen Fälle            #
###############################################################

df_chats = df_all_chats[df_all_chats["teilnehmer_id"].astype(int).isin(valid_cases)].copy()
df_chats["chat_uid"] = df_chats["teilnehmer_id"] + "_" + df_chats["frage_code"]
df_chats["chat_id"]  = df_chats.groupby("chat_uid").ngroup() + 1
df_chats = df_chats.sort_values(["chat_id", "turn"]).reset_index(drop=True)

df_chats.to_csv(path_proc / "chats_long.csv", index=False)
print(f"\nChat-Nachrichten: {len(df_chats)} | eindeutige Chats: {df_chats['chat_id'].nunique()} "
      f"(erwartet: {len(valid_cases) * 5})")
print(len(df_raw))

Survey bereinigt: 141 -> 101 Spalten (40 entfernt)
CASE 257: CH02s aus Mail übernommen (69 KB)
CASE 257: CH08s aus Mail übernommen (56 KB)
CASE 257: CH09s aus Mail übernommen (50 KB)
CASE 257: CH06s aus Mail übernommen (43 KB)
WARNUNG: CASE 255 CH03: HTML enthält keine Chat-Nachrichten (vermutlich leere ChatGPT-Seite gespeichert) — CH03.000255.html
WARNUNG: CASE 263 CH03: HTML enthält keine Chat-Nachrichten (vermutlich leere ChatGPT-Seite gespeichert) — CH03.000263.html
WARNUNG: CASE 264 CH03: HTML enthält keine Chat-Nachrichten (vermutlich leere ChatGPT-Seite gespeichert) — CH03.000264.html
WARNUNG: CASE 296 CH03: HTML enthält keine Chat-Nachrichten (vermutlich leere ChatGPT-Seite gespeichert) — CH03.000296.html
WARNUNG: CASE 368 CH03: HTML enthält keine Chat-Nachrichten (vermutlich leere ChatGPT-Seite gespeichert) — CH03.000368.html
WARNUNG: CASE 255 CH10: HTML enthält keine Chat-Nachrichten (vermutlich leere ChatGPT-Seite gespeichert) — CH10.000255.html
WARNUNG: CASE 263 CH10: HTML 

In [4]:
df_chats   = pd.read_csv(path_proc / "chats_long.csv")

df_labeled = cc.classify_chats(
    df_chats,
    model="gpt-5.5",
    api_key_path=repo / "API_KEYS/openai_key.txt",
    cache_path=path_proc / "classify_cache.json",  # bereits klassifizierte Chats werden nicht erneut angefragt
)

df_labeled.to_csv(path_proc / "chats_labeled.csv", index=False)
df_labeled

    10/105 Chats klassifiziert ...
    20/105 Chats klassifiziert ...
    30/105 Chats klassifiziert ...
    40/105 Chats klassifiziert ...
    50/105 Chats klassifiziert ...
    60/105 Chats klassifiziert ...
    70/105 Chats klassifiziert ...
    80/105 Chats klassifiziert ...
    90/105 Chats klassifiziert ...
    100/105 Chats klassifiziert ...
Fertig: 105 Chats klassifiziert, davon 0 aus dem Cache.


,chat_id,teilnehmer_id,frage_code,task,sentiment,critical,chat_text,raw_response
0,1,241,CH02,Technische und analytische Unterstützung,Neutral,Ja,User_Nachricht_1: I want to calculate the McDo...,TASK: Technische und analytische Unterstützung...
1,2,241,CH06,Technische und analytische Unterstützung,Neutral,Ja,User_Nachricht_1: I need to set some environme...,TASK: Technische und analytische Unterstützung...
2,3,241,CH07,Technische und analytische Unterstützung,Neutral,Nein,User_Nachricht_1: I have Images with multiple ...,TASK: Technische und analytische Unterstützung...
3,4,241,CH08,Technische und analytische Unterstützung,Neutral,Nein,User_Nachricht_1: I have this function in R. I...,TASK: Technische und analytische Unterstützung...
4,5,241,CH09,Technische und analytische Unterstützung,Freundlich,Nein,User_Nachricht_1: Please interpret this result...,TASK: Technische und analytische Unterstützung...
...,...,...,...,...,...,...,...,...
100,101,367,CH02,Technische und analytische Unterstützung,Neutral,Nein,User_Nachricht_1: why does this not work: \nva...,TASK: Technische und analytische Unterstützung...
101,102,367,CH06,Schreiben und Textarbeit,Neutral,Nein,User_Nachricht_1: give me suitable tags for Re...,TASK: Schreiben und Textarbeit\nSENTIMENT: Neu...
102,103,367,CH07,Technische und analytische Unterstützung,Neutral,Nein,User_Nachricht_1: wieso funktioniert das nicht...,TASK: Technische und analytische Unterstützung...
103,104,367,CH08,Technische und analytische Unterstützung,Neutral,Nein,User_Nachricht_1: das will ich ins forum schre...,TASK: Technische und analytische Unterstützung...


In [7]:
df_labeled.to_csv(path_proc / "chats_labeled.csv", index=False)

In [ ]:
######################################################
# Zufallsstichprobe: 30 % der klassifizierten Chats  #
# (feste Ziehung über seed=404, reproduzierbar)      #
######################################################

df_labeled_sample = (df_labeled
                     .sample(frac=0.3, random_state=404)
                     .sort_values("chat_id")
                     .reset_index(drop=True))

df_labeled_sample.to_csv(path_proc / "chats_labeled_sample30.csv", index=False)
print(f"Stichprobe: {len(df_labeled_sample)} von {len(df_labeled)} Chats (30 %, seed=404)")
df_labeled_sample

In [5]:
# Mapping der Label-Strings auf die Zielspalten-Namen (Rohcounts)
TASK_TO_COL = {
    "Informationssuche und Verständnis":            "obs_info_n",
    "Schreiben und Textarbeit":                     "obs_schreiben_n",
    "Praktische Unterstützung und Strukturierung":  "obs_praktisch_n",
    "Technische und analytische Unterstützung":     "obs_technisch_n",
    "Lernen und Prüfungsvorbereitung":              "obs_lernen_n",
}
SENT_TO_COL = {
    "Freundlich":   "obs_sent_freundlich_n",
    "Neutral":      "obs_sent_neutral_n",
    "Unfreundlich": "obs_sent_unfreundlich_n",
}
CRIT_TO_COL = {
    "Ja":   "obs_kritisch_ja_n",
    "Nein": "obs_kritisch_nein_n",
}

ALL_COUNT_COLS = (list(TASK_TO_COL.values())
                  + list(SENT_TO_COL.values())
                  + list(CRIT_TO_COL.values()))


def aggregate_to_person(df_labeled: pd.DataFrame,
                        drop_unknown: bool = True,
                        verbose: bool = True) -> pd.DataFrame:
    df = df_labeled.copy()

    # Optionale Kontrolle/Meldung zu unknown-Labels vor der Aggregation
    if verbose:
        for col in ["task", "sentiment", "critical"]:
            n_unknown = (df[col] == "unknown").sum()
            if n_unknown:
                print(f"WARNUNG: {n_unknown} Chats mit unknown in '{col}'")

    if drop_unknown:
        before = len(df)
        df = df[(df["task"] != "unknown") &
                (df["sentiment"] != "unknown") &
                (df["critical"] != "unknown")].copy()
        if verbose and before != len(df):
            print(f"{before - len(df)} Chats mit unknown-Labels entfernt.")

    # Label-Strings in Zielspalten-Namen übersetzen
    df["task_col"] = df["task"].map(TASK_TO_COL)
    df["sent_col"] = df["sentiment"].map(SENT_TO_COL)
    df["crit_col"] = df["critical"].map(CRIT_TO_COL)

    persons = sorted(df["teilnehmer_id"].unique())
    result = pd.DataFrame(0, index=persons, columns=ALL_COUNT_COLS, dtype=int)
    result.index.name = "teilnehmer_id"

    # Rohcounts je Person hochzählen
    for _, r in df.iterrows():
        result.loc[r["teilnehmer_id"], r["task_col"]] += 1
        result.loc[r["teilnehmer_id"], r["sent_col"]] += 1
        result.loc[r["teilnehmer_id"], r["crit_col"]] += 1

    # Anzahl gültiger Chats pro Person
    result.insert(0, "n_chats_valid",
                  df.groupby("teilnehmer_id").size().reindex(persons).fillna(0).astype(int))

    result = result.reset_index()

    # Konsistenzchecks: die drei Kategoriengruppen müssen je zu n_chats_valid summieren
    task_sum = result[list(TASK_TO_COL.values())].sum(axis=1)
    sent_sum = result[list(SENT_TO_COL.values())].sum(axis=1)
    crit_sum = result[list(CRIT_TO_COL.values())].sum(axis=1)
    assert (task_sum == result["n_chats_valid"]).all(), "Task-Counts != n_chats_valid"
    assert (sent_sum == result["n_chats_valid"]).all(), "Sentiment-Counts != n_chats_valid"
    assert (crit_sum == result["n_chats_valid"]).all(), "Kritik-Counts != n_chats_valid"

    if verbose:
        print(f"Aggregiert: {len(result)} Personen.")
    return result

df_person = aggregate_to_person(df_labeled)

Aggregiert: 21 Personen.


In [ ]:

# 1:1-Umbenennungen
RENAME = {
    "CASE":               "id",
    "DE01":               "gender",
    "DE05":               "degree",
    "DE06":               "field",
    "sd_argument":        "sd_1",
    "sd_stressed":        "sd_2",
    "sd_listening":       "sd_3",
    "sd_advantage":       "sd_4",
    "sd_litter":          "sd_5",
    "sd_help":            "sd_6",
    "NU01":               "ai_experience",
    "SC05":               "freq",                 # Nutzungshäufigkeit ChatGPT (SC04 = Filterfrage)
    "NU04_01":            "info_literacy_where",
    "NU04_02":            "info_literacy_how",
    "info_use_info":      "info_use_1",
    "info_use_text":      "info_use_2",
    "info_use_structure": "info_use_3",
    "info_use_tech":      "info_use_4",
    "info_use_study":     "info_use_5",
    "IP01":               "inter_style",
    "KR01":               "crit_visible_chat",
    "SE01_01":            "self_assess_1",
    "SE01_02":            "self_assess_2",
    "SE01_03":            "self_assess_3",
}

# KI-Tools (Mehrfachauswahl SC06, T/F): SC06_01 = ChatGPT (bei allen T),
# Item-Nummern 02/03/05/06 wie in der alten NU02-Frage
TOOL_MAP = {
    "SC06_02": "uses_gemini",
    "SC06_03": "uses_copilot",
    "SC06_05": "uses_deepseek",
    "SC06_06": "uses_claude",
}

MISSING_CODES = (-1, -9)


#SoSci-Mehrfachauswahl -> 0/1. Akzeptiert 'T'/'F' oder 2/1.
def _to_binary_tool(series: pd.Series) -> pd.Series:
    def conv(v):
        if isinstance(v, str):
            return 1 if v.strip().upper() == "T" else 0
        if pd.isna(v):
            return 0
        return 1 if int(v) == 2 else 0
    return series.map(conv)


def map_survey(df_raw: pd.DataFrame,
               recode_missing_to_na: bool = True) -> pd.DataFrame:
    df = df_raw.rename(columns={k: v for k, v in RENAME.items() if k in df_raw.columns}).copy()

    # age: Dropdown-Index -> echtes Alter (Index+15). Missing bleibt Missing.
    if "DE02" in df_raw.columns:
        idx = pd.to_numeric(df_raw["DE02"], errors="coerce")
        df["age"] = np.where(idx >= 3, idx + 15, np.nan)

    # KI-Tools binaer kodieren
    for src, tgt in TOOL_MAP.items():
        if src in df_raw.columns:
            df[tgt] = _to_binary_tool(df_raw[src])

    # Nur die relevanten Zielspalten behalten
    keep = ["id", "gender", "age", "degree", "field",
            "sd_1", "sd_2", "sd_3", "sd_4", "sd_5", "sd_6",
            "ai_experience", "uses_gemini", "uses_copilot", "uses_deepseek", "uses_claude",
            "freq", "info_literacy_where", "info_literacy_how",
            "info_use_1", "info_use_2", "info_use_3", "info_use_4", "info_use_5",
            "inter_style", "crit_visible_chat",
            "self_assess_1", "self_assess_2", "self_assess_3"]
    keep = [c for c in keep if c in df.columns]
    df = df[keep].copy()

    # Absicherung: diese Spalten braucht data_analasys.R (Abschnitt 3b)
    for c in ("freq", "info_literacy_where", "info_literacy_how"):
        assert c in df.columns, f"Spalte '{c}' fehlt — Quellvariable im SoSci-Export umbenannt?"

    # SoSci-Missing-Codes (-1/-9) optional zu NA (age ausgenommen: -1 war schon behandelt)
    if recode_missing_to_na:
        for c in df.columns:
            if c in ("id", "age"):
                continue
            if pd.api.types.is_numeric_dtype(df[c]):
                df[c] = df[c].replace(list(MISSING_CODES), np.nan)

    df["id"] = df["id"].astype(int)
    return df


#mergerd suvery-Daten mit aggregierten Chatlog-Counts
def merge_survey_chatlogs(df_survey_mapped: pd.DataFrame,
                          df_person_counts: pd.DataFrame,
                          how: str = "inner") -> pd.DataFrame:
    counts = df_person_counts.copy()
    counts["id"] = counts["teilnehmer_id"].astype(int)
    counts = counts.drop(columns=["teilnehmer_id"])

    merged = df_survey_mapped.merge(counts, on="id", how=how)
    return merged

df_survey_mapped = map_survey(df_raw)
df_final         = merge_survey_chatlogs(df_survey_mapped, df_person, how="inner")

df_final.to_csv(path_proc / "perp_dataset.csv", index=False)
df_final